In [2]:
project_root_path = '${PROJECT_ROOT_PATH}'
# @launchit.disable
project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]
# @launchit.stop

import sys
import os
import json
import multiprocessing as mp

import optuna
from optuna.trial import TrialState
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data
from torchvision import datasets
from torchvision import transforms

sys.path.append(project_root_path)
import lib.launchit # @launchit.disable
import lib.optuna_multiprocessing 

In [3]:
# @launchit.disable
with open(get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as cf:
    notebook_fname = json.load(cf)['jupyter_session']
    notebook_basename = os.path.basename(notebook_fname)
    notebook_name, notebook_ext = os.path.splitext(notebook_basename)

notebook_fname

'/home/misha/dev/mine/neurovision/experiment/optuna/optuna_torch_multiproc_test2.ipynb'

In [4]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCHSIZE = 128
CLASSES = 10
DIR = os.path.join(project_root_path, 'data', 'mnist')
EPOCHS = 10
N_TRAIN_EXAMPLES = BATCHSIZE * 30
N_VALID_EXAMPLES = BATCHSIZE * 10
STUDY_NAME = 'optuna_torch_miltiproc_test'

RUN_DIR = os.path.join(project_root_path, 'run', 'experiment', 'optuna')
os.makedirs(RUN_DIR, exist_ok=True)

STUDY_FNAME = os.path.join(RUN_DIR, STUDY_NAME + '.log')

In [5]:
def define_model(trial):
    # We optimize the number of layers, hidden units and dropout ratio in each layer.
    n_layers = trial.suggest_int("n_layers", 1, 3)
    layers = []

    in_features = 28 * 28
    for i in range(n_layers):
        out_features = trial.suggest_int("n_units_l{}".format(i), 4, 128)
        layers.append(nn.Linear(in_features, out_features))
        layers.append(nn.ReLU())
        p = trial.suggest_float("dropout_l{}".format(i), 0.2, 0.5)
        layers.append(nn.Dropout(p))

        in_features = out_features
    layers.append(nn.Linear(in_features, CLASSES))
    layers.append(nn.LogSoftmax(dim=1))

    return nn.Sequential(*layers)

In [6]:
def get_mnist():
    # Load FashionMNIST dataset.
    train_loader = torch.utils.data.DataLoader(
        datasets.FashionMNIST(DIR, train=True, download=True, transform=transforms.ToTensor()),
        batch_size=BATCHSIZE,
        shuffle=True,
    )
    valid_loader = torch.utils.data.DataLoader(
        datasets.FashionMNIST(DIR, train=False, transform=transforms.ToTensor()),
        batch_size=BATCHSIZE,
        shuffle=True,
    )

    return train_loader, valid_loader

In [6]:
trial = lib.optuna_multiprocessing.exchange.trial

# Generate the model.
model = define_model(trial).to(DEVICE)

# Generate the optimizers.
optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop", "SGD"])
lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
optimizer = getattr(optim, optimizer_name)(model.parameters(), lr=lr)

# Get the FashionMNIST dataset.
train_loader, valid_loader = get_mnist()

# Training of the model.
for epoch in range(EPOCHS):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        # Limiting training data for faster epochs.
        if batch_idx * BATCHSIZE >= N_TRAIN_EXAMPLES:
            break

        data, target = data.view(data.size(0), -1).to(DEVICE), target.to(DEVICE)

        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()

    # Validation of the model.
    model.eval()
    correct = 0
    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(valid_loader):
            # Limiting validation data.
            if batch_idx * BATCHSIZE >= N_VALID_EXAMPLES:
                break
            data, target = data.view(data.size(0), -1).to(DEVICE), target.to(DEVICE)
            output = model(data)
            # Get the index of the max log-probability.
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    accuracy = correct / min(len(valid_loader.dataset), N_VALID_EXAMPLES)

    trial.report(accuracy, epoch)

    # Handle pruning based on the intermediate value.
    if trial.should_prune():
        raise optuna.exceptions.TrialPruned()

lib.optuna_multiprocessing.exchange.result = accuracy

In [11]:
# @launchit.disable
launch_fname = lib.launchit.launchit(notebook_fname, make_py_file=True, expandvars={'PROJECT_ROOT_PATH': project_root_path}, dir_name=RUN_DIR, verbosity=2)
mp_ctx = mp.get_context('spawn')

with mp_ctx.Pool(processes=4) as pool:
    rop_list = [lib.optuna_multiprocessing.RunOptimizationParameters(
        module_fname=launch_fname,
        study_name=STUDY_NAME,
        direction='maximize',
        journal_fname=STUDY_FNAME,
        verbosity=2) for i in range(10)]
    pool.map(lib.optuna_multiprocessing.run_optimization, rop_list)

# join and report
study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=JournalStorage(JournalFileBackend(file_path=STUDY_FNAME)),
    load_if_exists=True, # Useful for multi-process or multi-node optimization.
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

print("Study statistics: ")
print("  Number of finished trials: ", len(study.trials))
print("  Number of pruned trials: ", len(pruned_trials))
print("  Number of complete trials: ", len(complete_trials))

print("Best trial:")
trial = study.best_trial

print("  Value: ", trial.value)

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

Creating /home/misha/dev/mine/neurovision/run/experiment/optuna/optuna_torch_multiproc_test2-launch4.py
Created "/home/misha/dev/mine/neurovision/run/experiment/optuna/optuna_torch_multiproc_test2-launch4.py"


[I 2026-01-25 11:34:41,110] A new study created in Journal with name: optuna_torch_miltiproc_test
[I 2026-01-25 11:34:41,131] Using an existing study with name 'optuna_torch_miltiproc_test' instead of creating a new one.
[I 2026-01-25 11:34:41,162] Using an existing study with name 'optuna_torch_miltiproc_test' instead of creating a new one.
[I 2026-01-25 11:34:41,181] Using an existing study with name 'optuna_torch_miltiproc_test' instead of creating a new one.


KeyboardInterrupt: 